# Bài 9 v4 ·PCA vàảnh thời trang
PGS.TS. Nguyễn Văn Hậu. Chạy từ đầu đến cuối. Bốn điểm là mô phỏng;Fashion-MNIST là dữ liệu thật. Không dùng metric test để chọn k.

In [1]:
from pathlib import Path
import sys,json
ROOT=Path.cwd()
if not (ROOT/"source/pca_lab.py").exists():raise RuntimeError("Mở notebook trong thư mục bài")
if (ROOT/".build/pydeps").exists():sys.path.insert(0,str(ROOT/".build/pydeps"))
sys.path.insert(0,str(ROOT/"source"))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from pca_lab import P,main
from threadpoolctl import threadpool_limits
from sklearn.decomposition import PCA
OUT=ROOT/"results/lab_figures";OUT.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({"figure.figsize":(10,5),"axes.grid":True})
print("PCA: tâm và hướng train; validation dùng lại tham số")

PCA: tâm và hướng train; validation dùng lại tham số


## 1. Phép chiếu tính tay
Hướng có độ dài 1,score là vị trí có dấu.

In [2]:
mu=P.mean(0);xc=P-mu;w=np.array([1,1])/np.sqrt(2);z=xc@w;rec=mu+z[:,None]*w;r=P-rec
assert np.allclose(r@w,0)
print(pd.DataFrame({"Điểm":list("ABCD"),"score":z,"x_hat1":rec[:,0],"x_hat2":rec[:,1],"error_point":(r*r).sum(1)}))
assert np.isclose(np.mean(r*r),.25)
print("MSE mỗi tọa độ",np.mean(r*r))

  Điểm    score  x_hat1  x_hat2  error_point
0    A -2.12132     3.5     3.5          0.5
1    B -2.12132     3.5     3.5          0.5
2    C  2.12132     6.5     6.5          0.5
3    D  2.12132     6.5     6.5          0.5
MSE mỗi tọa độ 0.25


## 2. Hình phép chiếu
Cùng trục, cùng tỷ lệ. Các điểm đứng yên, mũi tên nối đến chân chiếu.

In [3]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,points,reconstructed,origin in zip(axes,[P,xc],[rec,rec-mu],[mu,[0,0]]):
    ax.scatter(points[:,0],points[:,1],color="#245a84");ax.scatter(reconstructed[:,0],reconstructed[:,1],color="#bc761a",marker="D")
    for label,pnt,q in zip("ABCD",points,reconstructed):ax.annotate(label,pnt);ax.plot([pnt[0],q[0]],[pnt[1],q[1]],color="#bf514b")
    oo=np.array(origin);ends=oo+np.array([[-3], [3]])*w
    ax.plot(ends[:,0],ends[:,1],color="#bc761a");ax.set_aspect("equal");ax.set(xlabel="x1",ylabel="x2")
axes[0].set(xlim=(0,10),ylim=(0,10),title="Hệ gốc");axes[1].set(xlim=(-3,3),ylim=(-3,3),title="Căn giữa")
fig.tight_layout();fig.savefig(OUT/"projection.png",dpi=180);plt.show()

C:\Users\nguye\AppData\Local\Temp\ipykernel_3892\4071939841.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/"projection.png",dpi=180);plt.show()


## 3. Hướng nào giữ biến thiên?
PCA tối đa phương sai score, hướng phải đơn vị.

In [4]:
theta=np.linspace(0,np.pi,181);C=xc.T@xc/4;directions=np.column_stack([np.cos(theta),np.sin(theta)])
variance=np.einsum("bi,ij,bj->b",directions,C,directions);error=5-variance
assert np.isclose(variance.max(),4.5)
fig,ax=plt.subplots();ax.plot(np.degrees(theta),variance,label="Retained variance");ax.plot(np.degrees(theta),error,label="Mean squared point error");ax.axhline(5,color="gray",label="Total variance");ax.set(xlabel="Angle (degree)",ylabel="Population variance / squared error");ax.legend();fig.tight_layout();fig.savefig(OUT/"direction_choice.png",dpi=180);plt.show()
p=PCA(svd_solver="full").fit(P);assert np.allclose(p.explained_variance_,[6,2/3]);print("sklearn covariance sample:",p.explained_variance_,p.explained_variance_ratio_)

sklearn covariance sample: [6.         0.66666667] [0.9 0.1]


C:\Users\nguye\AppData\Local\Temp\ipykernel_3892\3180106826.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();ax.plot(np.degrees(theta),variance,label="Retained variance");ax.plot(np.degrees(theta),error,label="Mean squared point error");ax.axhline(5,color="gray",label="Total variance");ax.set(xlabel="Angle (degree)",ylabel="Population variance / squared error");ax.legend();fig.tight_layout();fig.savefig(OUT/"direction_choice.png",dpi=180);plt.show()


## 4. Tái lập từ dữ liệu gốc
Ô này fit PCA và hai Logistic Regression, chỉ đánh giá validation. Có thể mất vài phút.

In [5]:
with threadpool_limits(limits=2):main()
audit=json.loads((ROOT/"results/case_audit.json").read_text(encoding="utf8"))
assert audit["status"]=="pass"
arrays=np.load(ROOT/"data/lab_arrays.npz");data=np.load(ROOT/"data/fashion_subset.npz")
print("Kiểm case",audit["checks_count"],"k90",audit["k90"])
print(pd.DataFrame(audit["reconstruction"]))

Full PCA fitted 5.11 seconds


Validation logistic 0.8333333333333334 0.837


{"status": "pass", "checks_count": 45, "k90": 82, "duplicate_audit": {"train_validation_unique_hashes": 0, "train_test_unique_hashes": 0, "validation_test_unique_hashes": 0, "within_train_duplicate_rows": 0, "within_validation_duplicate_rows": 0, "within_test_duplicate_rows": 0}, "reconstruction": [{"k": 2, "train_evr": 0.46830407865190227, "train_mse_pixel": 0.0463744081993328, "validation_mse_pixel": 0.04661844525914814, "validation_outside_display_range_fraction": 0.12251275510204082, "validation_mse_clipped": 0.04616278455681625}, {"k": 16, "train_evr": 0.7659102095648508, "train_mse_pixel": 0.020417263065346462, "validation_mse_pixel": 0.02070311034786639, "validation_outside_display_range_fraction": 0.22235076530612244, "validation_mse_clipped": 0.020054270889954845}, {"k": 64, "train_evr": 0.8824213561722076, "train_mse_pixel": 0.010255184975970858, "validation_mse_pixel": 0.010561734931280922, "validation_outside_display_range_fraction": 0.24905442176870748, "validation_mse_cli

## 5. Theo dấu ảnh thật
ID gọi tên nguồn; vai trò validation. Patch là độ sáng pixel.

In [6]:
i=int(arrays["display_positions"][0]);image=data["validation_images"][i].reshape(28,28)
assert data["validation_indices"][i]==24814 and image[12,12]==203
print("ID train:24814; role validation; position",i)
print(pd.DataFrame(image[12:17,12:17],index=range(12,17),columns=range(12,17)))
print("Ba score đầu",arrays["validation_scores"][i,:3])
plt.imshow(image,cmap="gray",vmin=0,vmax=255);plt.title("train:24814 ·validation");plt.colorbar(label="Pixel byte");plt.grid(False);plt.savefig(OUT/"trace_image.png",dpi=180);plt.show()

ID train:24814; role validation; position

 3
     12   13   14   15   16
12  203  181  186  158  142
13  175  181  113  137  148
14  233  112   79  141  121
15  189   97  162  122   60
16   99  131  109  123  176
Ba score đầu [-2.58595193 -3.41366587  0.46329333]


C:\Users\nguye\AppData\Local\Temp\ipykernel_3892\2526453167.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(image,cmap="gray",vmin=0,vmax=255);plt.title("train:24814 ·validation");plt.colorbar(label="Pixel byte");plt.grid(False);plt.savefig(OUT/"trace_image.png",dpi=180);plt.show()


## 6.Cùngảnh,nhiều số chiều
MSE trước clip; hình clip [0,1] để hiển thị.

In [7]:
fig,axes=plt.subplots(1,5,figsize=(15,3));axes[0].imshow(image,cmap="gray",vmin=0,vmax=255);axes[0].set_title("Original")
for j,k in enumerate([2,16,64,128]):
    reconstruct=arrays["validation_scores"][i,:k]@arrays["components"][:k]+arrays["mean"]
    assert np.allclose(reconstruct,arrays["display_recon"][j,0])
    axes[j+1].imshow(np.clip(reconstruct.reshape(28,28),0,1),cmap="gray",vmin=0,vmax=1);axes[j+1].set_title("k="+str(k))
for ax in axes:ax.axis("off")
fig.tight_layout();fig.savefig(OUT/"reconstruction.png",dpi=180);plt.show()
print("Số tọa độ64/784:",64/784,"không là byte compression rate")

Số tọa độ64/784: 0.08163265306122448 không là byte compression rate


C:\Users\nguye\AppData\Local\Temp\ipykernel_3892\3129547784.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/"reconstruction.png",dpi=180);plt.show()


## 7. EVR và MSE trả lời câu hỏi khác nhau
Ngưỡng 90% dùng train; MSE dùng validation.

In [8]:
cum=arrays["cumulative_evr"];assert cum[80]<.9<=cum[81]
fig,axes=plt.subplots(1,2,figsize=(12,4));axes[0].plot(np.arange(1,201),cum[:200]);axes[0].axhline(.9,color="#bc761a");axes[0].set(xlabel="k",ylabel="Cumulative train EVR")
axes[1].plot([x["k"] for x in audit["reconstruction"]],[x["validation_mse_pixel"] for x in audit["reconstruction"]],"o-");axes[1].set(xlabel="k",ylabel="Validation MSE per pixel (unclipped)")
fig.tight_layout();fig.savefig(OUT/"dimension_evidence.png",dpi=180);plt.show()
print("81/82:",cum[80:82])

81/82: [0.8992718 0.9001215]


C:\Users\nguye\AppData\Local\Temp\ipykernel_3892\2086979216.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/"dimension_evidence.png",dpi=180);plt.show()


## 8. Hình 2D và pipeline phân loại
Nhãn tô màu để diễn giải, không fit hướng PCA.

In [9]:
scores=arrays["validation_scores"][:1000,:2];labels=data["validation_labels"][:1000]
fig,ax=plt.subplots(figsize=(7,7));ax.scatter(scores[:,0],scores[:,1],c=labels,cmap="tab10",s=10,alpha=.7);ax.set(xlim=(-15,15),ylim=(-15,15),xlabel="PC1 score",ylabel="PC2 score",title="Fixed first1000 validation images");ax.set_aspect("equal");fig.tight_layout();fig.savefig(OUT/"scores2d.png",dpi=180);plt.show()
y=data["validation_labels"];p=arrays["validation_pixel_predictions"]==y;q=arrays["validation_pca64_predictions"]==y
print(pd.DataFrame([[sum(p&q),sum(p&~q)],[sum(~p&q),sum(~p&~q)]],index=["Pixel correct","Pixel wrong"],columns=["PCA correct","PCA wrong"]))
assert sum(~p&q)==145 and sum(p&~q)==134
print("Net11 /3000; one split does not establish general superiority")

               PCA correct  PCA wrong
Pixel correct         2366        134
Pixel wrong            145        355
Net11 /3000; one split does not establish general superiority


C:\Users\nguye\AppData\Local\Temp\ipykernel_3892\945395686.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(figsize=(7,7));ax.scatter(scores[:,0],scores[:,1],c=labels,cmap="tab10",s=10,alpha=.7);ax.set(xlim=(-15,15),ylim=(-15,15),xlabel="PC1 score",ylabel="PC2 score",title="Fixed first1000 validation images");ax.set_aspect("equal");fig.tight_layout();fig.savefig(OUT/"scores2d.png",dpi=180);plt.show()


## 9. Kiểm khoa học và tự đánh giá
Thử chọn chiều theo mục đích,nêu hai bằng chứng vàmột giới hạn.

In [10]:
from science_audit import main as independent_audit
independent_audit()
print("EVR cao không bảo đảm dự đoán tốt: nhãn theo dấuPC2 của toy")
print("Gợi ý thử: giữk16/64/128 vàso cùngprotocol; không chọn bằngtest")

{"status": "pass", "checks_count": 181}
EVR cao không bảo đảm dự đoán tốt: nhãn theo dấuPC2 của toy
Gợi ý thử: giữk16/64/128 vàso cùngprotocol; không chọn bằngtest
